# 03. 머신러닝 모델 학습

## 중요: 연구 목적 재확인

> **본 연구의 목적은 등급 '예측'이 아님!**
>
> ML 모델은 XAI(SHAP) 분석을 위한 **도구**로 사용됨.
> 모델의 정확도 극대화보다 **해석 가능성(interpretability)**을 우선시.
>
> - 기존 연구: 모델을 학습시켜 등급 예측 → Accuracy 보고
> - 본 연구: 모델을 학습시켜 SHAP 값 추출 → **어떤 카테고리가 영향을 주었는가** 분석

## 학습 프로세스
```
[표준화된 v5 데이터]
        ↓
[Feature 구성]
  - v5 환산 카테고리 점수 9개 (LT, SS, WE, EA, MR, IEQ, IN, RP, IP)
  - 연면적 (log 변환)
  - 건물 유형 (원-핫 인코딩)
        ↓
[모델 선택 - 교차 검증]
  Random Forest / XGBoost / LightGBM
        ↓
[최종 모델 학습]
        ↓
[04번 노트북: SHAP 분석]
```

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

from src.data.loader import LEEDDataLoader
from src.data.preprocessor import LEEDPreprocessor, GRADE_ENCODING
from src.analysis.ml_models import LEEDMLTrainer

print('라이브러리 로딩 완료')

## 1. 표준화 데이터 로딩

In [ ]:
import os

standardized_path = '../data/processed/korea_leed_standardized_v5.csv'

if os.path.exists(standardized_path):
    # ─────────────────────────────────────────────────────────────────
    # 02번 노트북에서 생성한 표준화 데이터 사용
    # ─────────────────────────────────────────────────────────────────
    df = pd.read_csv(standardized_path)
    print(f'표준화 데이터 로딩: {len(df)}개 프로젝트')
else:
    # 파일이 없으면 다시 생성
    print('[주의] 표준화 파일 없음. 샘플 데이터 생성...')
    loader = LEEDDataLoader()
    preprocessor = LEEDPreprocessor()
    raw_df = loader.create_sample_data()
    df = preprocessor.run_pipeline(raw_df)

print(f'컬럼 수: {len(df.columns)}')
df.head(3)

## 2. Feature / Target 분리

In [ ]:
preprocessor = LEEDPreprocessor()

# grade_encoded 컬럼이 없으면 추가
if 'grade_encoded' not in df.columns:
    df = preprocessor.encode_grade(df)

# log_area 없으면 추가
if 'log_area' not in df.columns:
    df = preprocessor.log_transform_area(df)

# 건물 유형 원-핫 없으면 추가
if not any(c.startswith('type_') for c in df.columns):
    df = preprocessor.encode_building_type(df)

X, y = preprocessor.split_features_target(df)
feature_names = list(X.columns)

print(f'\nFeature 목록:')
for f in feature_names:
    print(f'  - {f}')

## 3. 상관관계 분석

In [ ]:
# v5 카테고리 점수 간 상관관계
v5_score_cols = [c for c in X.columns if c.startswith('score_v5_')]
corr_data = X[v5_score_cols].copy()
corr_data['grade'] = y.values

# 컬럼명 한국어 변환
kor_names = {
    'score_v5_LT': '입지교통(LT)',
    'score_v5_SS': '지속가능부지(SS)',
    'score_v5_WE': '물효율(WE)',
    'score_v5_EA': '에너지(EA)',
    'score_v5_MR': '재료자원(MR)',
    'score_v5_IEQ': '실내환경(IEQ)',
    'score_v5_IN': '혁신(IN)',
    'score_v5_RP': '지역우선(RP)',
    'score_v5_IP': '통합프로세스(IP)',
    'grade': '등급',
}
corr_data.rename(columns=kor_names, inplace=True)

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(
    corr_data.corr(),
    annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, ax=ax, square=True,
)
ax.set_title('LEED 카테고리별 점수 & 등급 상관관계\n(v5 기준 표준화 데이터)', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/03_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 모델 교차 검증 및 선택

In [ ]:
trainer = LEEDMLTrainer(output_dir='../outputs')

# 5-Fold 교차 검증으로 최적 모델 선택
cv_results = trainer.evaluate_models(X, y, cv_folds=5)
print('\n모델 성능 비교:')
display(cv_results)

## 5. 최종 모델 학습

In [ ]:
# 최고 성능 모델로 전체 데이터 학습
best_model = trainer.train(X, y)  # best_model_name 자동 사용

# 학습 데이터 기준 성능 확인 (참고용)
eval_results = trainer.evaluate(X, y)

# 모델 저장
trainer.save_model('leed_model.pkl')

## 6. 등급별 Feature 분포 시각화

In [ ]:
from src.data.preprocessor import GRADE_ENCODING
GRADE_LABELS = {v: k for k, v in GRADE_ENCODING.items()}

# EA(에너지) 점수 분포 - 등급별 비교
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. EA 점수 박스플롯
ea_data = []
for grade_enc, grade_name in GRADE_LABELS.items():
    mask = y == grade_enc
    if mask.sum() > 0 and 'score_v5_EA' in X.columns:
        ea_data.append(X.loc[mask, 'score_v5_EA'].values)

if ea_data:
    bp = axes[0].boxplot(ea_data, labels=[GRADE_LABELS[i] for i in sorted(GRADE_LABELS.keys()) if (y == i).sum() > 0],
                         patch_artist=True)
    colors = ['#e8e8e8', '#C0C0C0', '#FFD700', '#E5E4E2']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    axes[0].set_title('등급별 에너지·대기(EA) 점수 분포', fontsize=12)
    axes[0].set_ylabel('EA 점수 (v5 기준)')

# 2. 총점 분포
if 'total_score_v5' in df.columns:
    for grade_enc, grade_name in GRADE_LABELS.items():
        mask = df['grade_encoded'] == grade_enc
        if mask.sum() > 0:
            df.loc[mask, 'total_score_v5'].hist(
                bins=20, ax=axes[1], alpha=0.6, label=grade_name
            )
    axes[1].set_title('등급별 v5 환산 총점 분포', fontsize=12)
    axes[1].set_xlabel('v5 총점')
    axes[1].set_ylabel('건물 수')
    axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/03_grade_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n03번 노트북 완료. 다음: 04-XAI-SHAP-Analysis.ipynb')